##### Import the libraries

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import holidays
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score)
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import joblib
import os
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

##### Load the datasets

In [31]:
predictions = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Best Models\Best Models Predictions\best_models_predictions.csv")

predictions["date"] = pd.to_datetime(predictions["date"])

predictions.head()

,hospital_name,hospital_id,unit,date,actual,predicted,model
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-25,21.208333,22.281654,SARIMA
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-26,23.375000,21.692058,SARIMA
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-27,19.000000,19.730131,SARIMA
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-28,18.875000,17.138078,SARIMA
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-29,16.916667,16.988289,SARIMA


##### Calculate Predictions Errors

In [32]:
predictions["error"] = (
    predictions["actual"]
    - predictions["predicted"]
)

predictions["absolute_error"] = (
    predictions["error"].abs()
)

predictions["percentage_error"] = (
    predictions["absolute_error"]
    / predictions["actual"]
) * 100

##### Overall performance

In [33]:
overall_mae = mean_absolute_error(
    predictions["actual"],
    predictions["predicted"]
)

overall_rmse = np.sqrt(
    mean_squared_error(
        predictions["actual"],
        predictions["predicted"]
    )
)

overall_mape = predictions["percentage_error"].mean()

print("="*40)
print("OVERALL MODEL PERFORMANCE")
print("="*40)

print(f"MAE  : {overall_mae:.3f}")
print(f"RMSE : {overall_rmse:.3f}")
print(f"MAPE : {overall_mape:.2f}%")

OVERALL MODEL PERFORMANCE
MAE  : 1.166
RMSE : 2.026
MAPE : inf%


##### Calculate monitoring metrics (Performance by Hospital Unit)

In [34]:
monitoring_results = (predictions.groupby([
            "hospital_name",
            "hospital_id",
            "unit",
            "model"]).apply(lambda x: pd.Series({
                "MAE":mean_absolute_error(x["actual"], x["predicted"]),
                "RMSE":np.sqrt(mean_squared_error(x["actual"], x["predicted"])),
                "MAPE":(np.abs((x["actual"] - x["predicted"])/ x["actual"]).mean())*100,
            "Mean Actual": x["actual"].mean(),
            "Mean Prediction": x["predicted"].mean(),
            "Observations": len(x)})).reset_index())

monitoring_results.head()

,hospital_name,hospital_id,unit,model,MAE,RMSE,MAPE,Mean Actual,Mean Prediction,Observations
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359,18.380952,19.553162,7.0
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,inf,0.311111,0.382647,30.0
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993,15.839286,15.799775,7.0
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891,14.323611,14.501197,30.0
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822,8.672619,8.801221,7.0


##### Save monitoring results

In [35]:
monitoring_results.to_csv("../Monitoring/prediction_accuracy_monitoring.csv", index=False)
print("Prediction monitoring saved.")

Prediction monitoring saved.


##### Prediction error

In [36]:
predictions = predictions.sort_values(["unit", "date"])
predictions["rolling_mae"] = (predictions.groupby("unit")["absolute_error"].transform(
    lambda x: x.rolling(window=30, min_periods=7).mean()))

predictions.head()

,hospital_name,hospital_id,unit,date,actual,predicted,model,error,absolute_error,percentage_error,rolling_mae
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-25,21.208333,22.281654,SARIMA,-1.073320,1.073320,5.060842,NaN
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-26,23.375000,21.692058,SARIMA,1.682942,1.682942,7.199751,NaN
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-27,19.000000,19.730131,SARIMA,-0.730131,0.730131,3.842792,NaN
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-28,18.875000,17.138078,SARIMA,1.736922,1.736922,9.202237,NaN
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-29,16.916667,16.988289,SARIMA,-0.071622,0.071622,0.423383,NaN


##### Drift detection

In [37]:
baseline_error = (predictions.groupby("unit")["absolute_error"].transform("mean"))
predictions["baseline_error"] = baseline_error
predictions["drift_threshold"] = (predictions["baseline_error"] * 1.5)

predictions["drift_status"] = np.where(predictions["rolling_mae"] > predictions["drift_threshold"], "DRIFT DETECTED", "Normal")

drift_summary = (predictions.groupby(["hospital_name", "hospital_id", "unit"])
    .agg(Current_Rolling_MAE=("rolling_mae","last"), Baseline_Error=("baseline_error","last"), Drift_Threshold=("drift_threshold","last"),
        Drift_Status=("drift_status","last")).reset_index())

drift_summary.head()

,hospital_name,hospital_id,unit,Current_Rolling_MAE,Baseline_Error,Drift_Threshold,Drift_Status
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2.149314,2.149314,3.223971,Normal
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Day Case Unit,0.160609,0.160609,0.240914,Normal
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,0.216116,0.216116,0.324174,Normal
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,1.161449,1.161449,1.742174,Normal
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_ICU,0.284338,0.284338,0.426507,Normal


In [38]:
drift_summary.to_csv("../Monitoring/drift_detection_results.csv",index=False)

print("Drift report saved.")

Drift report saved.


##### Retraining Recommendation

In [39]:

MAPE_THRESHOLD = 15
drift_summary = drift_summary.merge(monitoring_results[[
            "hospital_name",
            "hospital_id",
            "unit",
            "MAPE"]],

    on=[
        "hospital_name",
        "hospital_id",
        "unit"],

    how="left")

drift_summary["Retraining_Recommendation"] = np.where((drift_summary["MAPE"] > MAPE_THRESHOLD)|
            (drift_summary["Drift_Status"] == "DRIFT DETECTED"), "RETRAIN MODEL", "MODEL OK")

drift_summary.head()

,hospital_name,hospital_id,unit,Current_Rolling_MAE,Baseline_Error,Drift_Threshold,Drift_Status,MAPE,Retraining_Recommendation
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2.149314,2.149314,3.223971,Normal,13.236359,MODEL OK
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Day Case Unit,0.160609,0.160609,0.240914,Normal,inf,RETRAIN MODEL
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,0.216116,0.216116,0.324174,Normal,1.365993,MODEL OK
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,1.161449,1.161449,1.742174,Normal,8.634891,MODEL OK
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_ICU,0.284338,0.284338,0.426507,Normal,3.549822,MODEL OK


##### Retraining Log

In [40]:
retraining_log = drift_summary.copy()

retraining_log["Review_Date"] = pd.Timestamp.today().normalize()

retraining_log = retraining_log[
    [
        "Review_Date",
        "hospital_name",
        "hospital_id",
        "unit",
        "MAPE",
        "Drift_Status",
        "Retraining_Recommendation"
    ]
]

retraining_log.head()

,Review_Date,hospital_name,hospital_id,unit,MAPE,Drift_Status,Retraining_Recommendation
0,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,13.236359,Normal,MODEL OK
1,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Day Case Unit,inf,Normal,RETRAIN MODEL
2,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,1.365993,Normal,MODEL OK
3,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,8.634891,Normal,MODEL OK
4,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_ICU,3.549822,Normal,MODEL OK


In [41]:
retraining_log.to_csv("../Monitoring/retraining_log.csv",index=False)

print("Retraining log saved.")

Retraining log saved.


##### Display final results

In [42]:
print("="*60)
print("CONTINUOUS LEARNING SUMMARY")
print("="*60)

print(f"Overall MAE  : {overall_mae:.3f}")
print(f"Overall RMSE : {overall_rmse:.3f}")
print(f"Overall MAPE : {overall_mape:.2f}%")

print()

print("Units requiring retraining:")

display(

    retraining_log[
        retraining_log["Retraining_Recommendation"]
        ==
        "RETRAIN MODEL"
    ]

)

CONTINUOUS LEARNING SUMMARY
Overall MAE  : 1.166
Overall RMSE : 2.026
Overall MAPE : inf%

Units requiring retraining:


,Review_Date,hospital_name,hospital_id,unit,MAPE,Drift_Status,Retraining_Recommendation
1,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Day Case Unit,inf,Normal,RETRAIN MODEL
5,2026-07-29,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Oncology Ward,16.813870,Normal,RETRAIN MODEL
9,2026-07-29,Horizon Edinburgh,HHN-EDI-01,HHN-EDI-01_Day Case Unit,inf,Normal,RETRAIN MODEL
16,2026-07-29,Horizon London Central,HHN-LON-01,HHN-LON-01_Cardiology Ward,15.226869,Normal,RETRAIN MODEL
17,2026-07-29,Horizon London Central,HHN-LON-01,HHN-LON-01_Day Case Unit,inf,Normal,RETRAIN MODEL
25,2026-07-29,Horizon London Riverside,HHN-LON-02,HHN-LON-02_Day Case Unit,inf,Normal,RETRAIN MODEL
33,2026-07-29,Horizon Manchester,HHN-MAN-01,HHN-MAN-01_Day Case Unit,inf,Normal,RETRAIN MODEL
